### Setup library

In [5]:
!pip install -q unsloth trl accelerate bitsandbytes datasets pandas

In [4]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-7b-bnb-4bit",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.10.1: Fast Gemma patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.57G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/154 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/34.4M [00:00<?, ?B/s]

### Process Dataset

In [7]:
import pandas as pd

df = pd.read_csv("/content/test_final.csv")
df = df.dropna(subset=["prompt", "essay", "band"])
df["band"] = df["band"].astype(str).str.strip().replace({"<4": "3.5"}).astype(float)


In [8]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are an IELTS Writing Task 2 examiner. Evaluate the essay using IELTS band descriptors.
Internally calculate scores for the four criteria, average them, and round to the nearest 0.5.
Return ONLY the overall band score in strict JSON format.

### Input:
Essay prompt: {}
Essay: {}

### Response:
{}"""


### Format to prompt response

In [9]:
import json

EOS_TOKEN = tokenizer.eos_token

def format_band_to_json(band_value):
    return json.dumps({"overallScore": band_value})

def formatting_prompts_func(examples):
    prompts = examples["prompt"]
    essays  = examples["essay"]
    outputs = examples["band"]
    texts = []
    for prompt, essay, output in zip(prompts, essays, outputs):
        output_json = format_band_to_json(output)
        text = alpaca_prompt.format(prompt, essay, output_json) + EOS_TOKEN
        texts.append(text)
    return { "text": texts }


### Create dataset

In [10]:
from datasets import Dataset

dataset = Dataset.from_pandas(df[["prompt", "essay", "band"]])
dataset = dataset.map(formatting_prompts_func, batched=True)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = dataset["train"]
eval_dataset  = dataset["test"]


Map:   0%|          | 0/495 [00:00<?, ? examples/s]

### LoRA

In [11]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = True,
    random_state = 42,
    use_rslora = False,
)


Unsloth 2025.10.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


### Config train

In [12]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "/content/genma_band_model",
        report_to = "none",
        eval_strategy = "steps",
        eval_steps = 100,
    ),
)


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/445 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/50 [00:00<?, ? examples/s]

In [21]:
trainer.train()


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 445 | Num Epochs = 3 | Total steps = 168
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 50,003,968 of 8,587,684,864 (0.58% trained)


Step,Training Loss,Validation Loss
100,1.013300,2.184157


TrainOutput(global_step=168, training_loss=1.0824554704484486, metrics={'train_runtime': 2596.7973, 'train_samples_per_second': 0.514, 'train_steps_per_second': 0.065, 'total_flos': 3.301746093138739e+16, 'train_loss': 1.0824554704484486, 'epoch': 3.0})

In [22]:
model.save_pretrained("/content/genma_band_model")
tokenizer.save_pretrained("/content/genma_band_model")


('/content/genma_band_model/tokenizer_config.json',
 '/content/genma_band_model/special_tokens_map.json',
 '/content/genma_band_model/tokenizer.model',
 '/content/genma_band_model/added_tokens.json',
 '/content/genma_band_model/tokenizer.json')

## infernce

In [23]:
from transformers import TextStreamer
import json
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

streamer = TextStreamer(tokenizer)
predicted_scores = []
true_scores = []

for example in eval_dataset:
    prompt = alpaca_prompt.format(example["prompt"], example["essay"], "")
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    output = model.generate(**inputs, max_new_tokens=128)
    decoded = tokenizer.decode(output[0], skip_special_tokens=True)

    try:
        json_str = decoded.split("### Response:")[-1].strip()
        band = json.loads(json_str)["overallScore"]
        predicted_scores.append(float(band))
        true_scores.append(float(example["band"]))
    except:
        continue


In [24]:
mae = mean_absolute_error(true_scores, predicted_scores)
rmse = np.sqrt(mean_squared_error(true_scores, predicted_scores))

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")

MAE: 0.98
RMSE: 1.33
